# ICT-46 — Strate 7 : freebits de second ordre, le banc exécutable
**Navigation** : [Index](README.md) | [45 — Inoculation & bifurcation 9B](ICT-42b-InoculationBifurcation-9B-Python.ipynb) | [47 — L'axe douleur](ICT-47-PainAxisDistillation-Python.ipynb)

**Première livraison de la strate 7** (issue [#18052](https://github.com/jsboige/CoursIA/issues/18052), veine 3 de l'Epic Aaronson [#16781](https://github.com/jsboige/CoursIA/issues/16781)). Le cadrage D1 [#7745](https://github.com/jsboige/CoursIA/issues/7745) posait l'objet — les *free coordinates* / freebits de second ordre — et ses six proxys mesurables, en prose ([`strate7-cadres-libres.md`](https://github.com/jsboige/CoursIA/blob/main/docs/ict/strate7-cadres-libres.md)). Ce notebook est l'organe : il fait **tourner** le contraste qui définit la strate.

**L'étage Aaronson (ordre 1).** Dans *The Ghost in the Quantum Turing Machine* (2013), Scott Aaronson propose le **freebit** : une incertitude *de Knight* — non réductible à une distribution de probabilité — sur la **valeur** d'une variable dans un espace **fixé**. Le cerveau amplifierait de tels bits quantiques microscopiques ; le libre arbitre y trouverait un ancrage *physique* dans les conditions aux limites de l'univers plutôt que dans les lois dynamiques. Aaronson se dit lui-même sceptique de son image (abstract : *"of which I myself remain skeptical"*), et la présente comme un exercice de *model-building* falsifiable.

**L'étage strate 7 (ordre 2).** Le cadrage ICT pose la question strictement plus forte : et si l'incertitude ne portait pas sur une *valeur dans un espace donné*, mais sur **l'espace lui-même** — le choix, non canonique, de l'extension par laquelle un système se dote de coordonnées qu'il n'avait pas ? C'est le saut du freebit d'Aaronson au freebit de second ordre.

**Ce que ce notebook démontre, de façon exécutable :**

1. un prédicteur d'espace fixé ne peut être *surpris structurellement* que par ce qui n'est pas dans son espace (§2) ;
2. le coup ontologique — ajouter une coordonnée — est mesurable par six proxys indépendants, dont trois au moins sont non triviaux ici (§3-§6) ;
3. les deux lectures du geste d'Aaronson coexistent avec leurs grades (§7), sous le contrat de falsifiabilité [#7291](https://github.com/jsboige/CoursIA/issues/7291).

Compagnon : module [`ict/free_coordinates.py`](ict/free_coordinates.py) (12 tests verts dans `tests/test_free_coordinates.py`).

## 1. Le banc : jeu évolutif $G_t$ et deux types de coups

Le cadrage D1 définit l'état de la strate 7 comme un **jeu évolutif** $G_t = (N, L_t, A_t, U_t)$ — agents, langage, actions, utilités. Deux types de coups : les **internes** $a \in A_t$ (le jeu ne bouge pas) et les **ontologiques** $\eta : G_t \to G_{t+1}$ (le jeu change d'espace). Nous instancions le banc minimal : deux agents, un lexique d'un concept, deux actions.

In [1]:
import sys
sys.path.insert(0, ".")
import numpy as np
import matplotlib.pyplot as plt
from ict.free_coordinates import (
    EvolutiveGame, OntologicalMove, internal_move, apply_ontology,
    quorum_mechanism, expansion_ontology, political_opening, non_canonicity,
    simulate_trajectory, performative_power, institutionalization,
    irreversibility_debt, FixedSpacePredictor,
)

G0 = EvolutiveGame(
    agents=("alice", "bob"),
    lexicon=frozenset({"echange"}),
    actions=frozenset({"vendre", "garder"}),
    payoffs={
        ("alice", "vendre"): 1.0, ("alice", "garder"): 0.4,
        ("bob", "vendre"): 0.3, ("bob", "garder"): 1.0,
    },
)
print("G0 :", len(G0.agents), "agents,", len(G0.lexicon), "concept(s),", len(G0.actions), "action(s)")

# Coup interne : le paiement est lu, l'espace ne bouge pas.
p = internal_move(G0, "alice", "vendre")
print(f"internal_move(alice, vendre) = {p} — G0 inchange : {len(G0.actions)} action(s)")

G0 : 2 agents, 1 concept(s), 2 action(s)
internal_move(alice, vendre) = 1.0 — G0 inchange : 2 action(s)


### Lecture du résultat

Le coup interne est la strate 6 : un agent *dans* un jeu fixé. Tout ce qui suit s'intéresse au moment où l'agent (ou le mécanisme) *transforme le jeu lui-même*.

## 2. L'étage Aaronson : un prédicteur d'espace fixé ne peut être surpris que par l'inconnu structurel

Le point technique d'Aaronson est subtil : même une incertitude *totale* sur la **valeur** d'une action se laisse réduire à une distribution — c'est le régime bayésien ordinaire, dont la log-perte est bornée par l'entropie. Le freebit échappe à cette réduction *par physique* (no-cloning, past hypertext), pas par probabilité. Notre banc matérialise la frontière : tant que l'action observée reste dans l'espace connu, le prédicteur apprend ; la **première** action hors de l'espace ne lui coûte pas « une grande perte » — elle révèle que son espace était le mauvais objet.

In [2]:
rng = np.random.default_rng(2026)

pred = FixedSpacePredictor(("vendre", "garder"))
perte_connue = []
for _ in range(200):
    a = "vendre" if rng.random() < 0.7 else "garder"
    pred.update(a)
    perte_connue.append(pred.log_loss(a, epsilon=1.0))

surprise = pred.log_loss("signer")  # hors de l'espace fixe

print(f"Perte sur support connu (moyenne, 30 derniers coups) : {np.mean(perte_connue[-30:]):.3f} bits")
print(f"Surprise structurelle pour 'signer' (hors espace)    : {surprise:.3f} bits  = log2(|A|+1) = log2(3)")
print(f"-> l'ecart n'est pas de volume, il est de NATURE : probabiliste vs structurel.")

Perte sur support connu (moyenne, 30 derniers coups) : 0.938 bits
Surprise structurelle pour 'signer' (hors espace)    : 1.585 bits  = log2(|A|+1) = log2(3)
-> l'ecart n'est pas de volume, il est de NATURE : probabiliste vs structurel.


### Lecture du résultat

Sur son support, le prédicteur converge — 0,938 bit par coup en moyenne, contre $H(0{,}7) \approx 0{,}881$ bit, l'entropie du tirage 70/30 (l'écart vient du lissage de Laplace de l'estimateur, qui tire les probabilités vers 1/2). La surprise hors support vaut $\log_2(|A|+1)$ : le +1 est **la porte que l'espace n'avait pas**. C'est la forme calculable de la différence entre *ignorer une valeur dans $L$* (freebit d'Aaronson, ordre 1) et *ignorer $L$ lui-même* (free coordinate, ordre 2) : la première incertitude se monnaie en bits de valeur, la seconde en bits de **coordonnée**.

## 3. Le coup ontologique : la « personne morale » comme ajout de coordonnée

L'exemple paradigmatique du cadrage D1 : **personne morale**. Zéro bit physique ajouté au monde ; une coordonnée efficace ajoutée au monde social — l'entreprise peut *signer* et *posséder*. Appliquons ce $\eta$ et mesurons les deux premiers proxys.

In [3]:
ETA_PM = OntologicalMove(
    proposer="alice",
    new_concepts=("personne-morale",),
    opened_actions={"personne-morale": ("signer", "posseder")},
    pose_cost=1.0, undo_cost=50.0,
)
G1 = apply_ontology(G0, ETA_PM)
print(f"O_t  (expansion ontologique) = {expansion_ontology(G0, G1)}")
print(f"dA_t (ouverture politique)   = {political_opening(G0, G1)}")

# Contrastant : le concept ornemental n'ouvre RIEN.
ETA_POESIE = OntologicalMove(proposer="bob", new_concepts=("poesie",), opened_actions={})
G_poesie = apply_ontology(G0, ETA_POESIE)
print(f"Poesie : O_t = {expansion_ontology(G0, G_poesie)}, dA_t = {political_opening(G0, G_poesie)} (verbiage detecte)")

O_t  (expansion ontologique) = 1
dA_t (ouverture politique)   = 2
Poesie : O_t = 1, dA_t = 0 (verbiage detecte)


### Lecture du résultat

$O_t$ compte les concepts nouveaux ; $\Delta A_t$ est le **garde-fou contre le verbiage** : un concept qui n'ouvre aucune action est décoratif — l'expansion ontologique ne compte que si elle *change ce que les agents peuvent faire*. La poésie ouvre ici zéro action : dans ce banc minimal, elle ne franchit pas la barrière de la strate 7.

## 4. Non-canonicité $C_t$ : y a-t-il quelque chose à choisir ?

Le proxy le plus directement lié à la grammaire extensionnelle : $C_t = |AdmExt(G_t)|$, le nombre d'extensions **non équivalentes** (deux extensions sont équivalentes si elles ouvrent les mêmes actions). Si $C_t = 1$, le prolongement est canonique — la strate 7 est triviale.

In [4]:
e_pm   = OntologicalMove("alice", ("personne-morale",), {"personne-morale": ("signer", "posseder")})
e_soc  = OntologicalMove("bob", ("societe",), {"societe": ("signer", "posseder")})
e_hyp  = OntologicalMove("bob", ("hypotheque",), {"hypotheque": ("nantir",)})

print(f"C_t(personne-morale seule)          = {non_canonicity(G0, (e_pm,))}")
print(f"C_t(+ societe, equivalente)         = {non_canonicity(G0, (e_pm, e_soc))}")
print(f"C_t(+ hypotheque, NON equivalente)  = {non_canonicity(G0, (e_pm, e_soc, e_hyp))}")

C_t(personne-morale seule)          = 1
C_t(+ societe, equivalente)         = 1
C_t(+ hypotheque, NON equivalente)  = 2


### Lecture du résultat

Ajouter une extension *équivalente* (la « société » ouvre les mêmes actions que la « personne morale », à renommage près) ne change pas $C_t$ : il n'y a toujours rien à choisir. Ajouter l'« hypothèque » — qui ouvre *nantir*, une action différente — fait passer $C_t$ à 2 : **il y a maintenant un choix, et ce choix est réel**. C'est la mesure $H^1 \neq 0$ du cadrage : l'obstruction est ce qu'il y a à choisir.

## 5. Pouvoir performatif $P(R)$ : le coup change-t-il la trajectoire du système ?

$P(R) = D(\mathrm{Pr}(traj \mid do(\eta)) \,\|\, \mathrm{Pr}(traj \mid do(\neg\eta)))$ — la divergence KL entre les distributions d'actions du système avec et sans le coup ontologique (interventionnisme de Pearl), estimée par Monte Carlo à **lissage de Laplace symétrique sur l'union des supports** et exprimée en **bits**. Le cadrage D1 pose au niveau formel : *un coup décoratif — un nouveau mot qui n'agit sur rien — a $P(R) \approx 0$*. Le banc rend cette définition opératoire, et honnêtement : une copie d'action au profil de paiements identique reste une option **distinguée** pour la dynamique (elle capte du temps de jeu) ; elle n'est décorative qu'**au quotient des équivalences** — la même equivalence action/classe que le $C_t$ du §4.

In [5]:
rng = np.random.default_rng(42)
G1_pm = apply_ontology(G0, ETA_PM)
P_pm = performative_power(G1_pm, G0, horizon=20, n_sim=8, rng=rng)

ETA_DECO = OntologicalMove("alice", ("ornement",), {"ornement": ("vendre2",)})
G_deco = apply_ontology(G0, ETA_DECO)
for agent in G_deco.agents:  # 'vendre2' paie exactement comme 'vendre' : copie decorative
    G_deco.payoffs[(agent, "vendre2")] = G_deco.payoffs[(agent, "vendre")]
P_deco = performative_power(G_deco, G0, horizon=20, n_sim=8, rng=rng)
P_deco_eq = performative_power(G_deco, G0, horizon=20, n_sim=8, rng=rng, equivalence={"vendre2": "vendre"})

print(f"P(R) personne-morale       = {P_pm:.3f} bits  (signer/posseder : de vraies nouvelles actions)")
print(f"P(R) decoratif brut        = {P_deco:.3f} bits  (vendre2 reste une option DISTINGUEE)")
print(f"P(R) decoratif marginalise = {P_deco_eq:.3f} bits  (vendre2 equivaut a vendre : dynamique de fond inchangee)")

P(R) personne-morale       = 2.753 bits  (signer/posseder : de vraies nouvelles actions)
P(R) decoratif brut        = 1.819 bits  (vendre2 reste une option DISTINGUEE)
P(R) decoratif marginalise = 0.029 bits  (vendre2 equivaut a vendre : dynamique de fond inchangee)


### Lecture du résultat

Trois nombres, trois lectures. La **personne morale** ($P(R) \approx 2{,}8$ bits) transforme la distribution des trajectoires : `signer` et `posseder` sont de nouvelles options, avec leurs paiements propres, qui captent du temps de jeu. La **copie décorative brute** garde un $P(R)$ non nul ($P(R) \approx 1{,}8$ bit) : `vendre2` est une option *distinguée* — le softmax lui consacre du temps de jeu que la baseline accorde à `vendre` seul. **Marginalisée par l'équivalence** `vendre2 ≡ vendre`, elle tombe à $\approx 0{,}03$ bit : la copie ne change pas la dynamique de *fond*, seulement son habillage. C'est la définition D1 rendue opératoire — *un nouveau mot qui n'agit sur rien* — et la même equivalence action/classe que le quotient d'extensions du $C_t$ (§4). Attention à la lecture : $P(R)$ mesure le *pouvoir causal du coup*, pas sa *légitimité* — un coup manipulateur a aussi un $P(R)$ élevé. Les proxys suivant précisent le destin temporel.

## 6. Institutionnalisation et dette d'irréversibilité $I(R)$

Le coup survit-il à son instigateur ? Et que coûte son défaire, relativement à sa pose ? Ce sont les deux proxys temporels : la **persistance après retrait** et l'**asymétrie création/dissolution**.

In [6]:
rng = np.random.default_rng(7)
rate_pm = institutionalization(G1_pm, ETA_PM, horizon=60, rng=rng)

ETA_JARGON = OntologicalMove("alice", ("jargon-prive",), {"jargon-prive": ("jargonner",)})
G_jargon = apply_ontology(G0, ETA_JARGON)
for agent in G_jargon.agents:
    G_jargon.payoffs[(agent, "jargonner")] = -2.0  # les autres EVITENT le jargon prive
rate_jargon = institutionalization(G_jargon, ETA_JARGON, horizon=60, rng=rng)

print(f"Institutionnalisation personne-morale = {rate_pm:.2%} (les autres continuent de l'utiliser)")
print(f"Institutionnalisation jargon-prive    = {rate_jargon:.2%} (meurt avec son inventeur)")
print(f"I(R) personne-morale = {irreversibility_debt(ETA_PM):.0f} x (defaire coute 50x poser)")

Institutionnalisation personne-morale = 41.67% (les autres continuent de l'utiliser)
Institutionnalisation jargon-prive    = 0.00% (meurt avec son inventeur)
I(R) personne-morale = 50 x (defaire coute 50x poser)


### Lecture du résultat

Le jargon privé — que les autres agents *évitent activement* — meurt avec son instigateur : institutionnalisation quasi nulle. La personne morale persiste chez les survivants. Et sa dette d'irréversibilité ($50\times$) matérialise la signature des *macro-faits* : **une extension qui réussit devient un fait stable — le free coordinate disparaît au moment même où il réussit**.

## 7. Les deux lectures, avec leurs grades

**Lecture 1 — celle du texte** *(grade : interprétation sourcée)*. Aaronson propose une image *physique* du libre arbitre : des freebits — bits quantiques microscopiques dont les valeurs, fixées par les conditions initiales de l'univers, sont *de Knight* (non assignables de distribution) pour tout observant — seraient amplifiés par la micro-physique cérébrale jusqu'à rendre nos actes in-principe imprévisibles, même par un simulateur parfait. Il la présente comme *model-building* : sensée (*"not obviously much crazier than the alternatives"*) et falsifiable — des questions empiriques ouvertes en neuroscience et en physique doivent trancher une manière plutôt qu'une autre.

**Lecture 2 — d'ordre 2, tenue contre elle** *(grade : garde-fou épistémique, posé avant tout résultat)*. Trois réserves accompagnent toute transposition : (i) l'objet qui monte en strate ici n'est pas le freebit physique mais le **choix d'extension d'espace** — un étage au-dessus, et le glissement d'un étage à l'autre doit être déclaré, pas naturalisé ; (ii) l'attachement du chercheur à ses propres formes est **enregistré d'avance comme biais possible** (le §218 décrit l'idéalisme qui « sélectionne les faits, les déforme, les renomme ») — se référer à soi-même est bon marché, bien choisir sa coordonnée libre ne l'est pas ; (iii) le contrat de falsifiabilité [#7291](https://github.com/jsboige/CoursIA/issues/7291) exige que l'objet ne monte **que si un résultat falsifiable l'exige** — les six proxys de ce notebook sont une *grille de dicibilité* (elle rend dicibles des non-résultats), pas une démonstration.

Les deux lectures coexistent : la première donne au mot sa source et sa physics ; la seconde encadre ce que le dépôt peut en revendiquer. Aucune n'efface l'autre.

## Encart gap Mathlib

Le banc repose sur : comptage d'ensembles finis ($O_t$, $\Delta A_t$, $C_t$), divergence KL empirique ($P(R)$), fractions de trajectoires (institutionnalisation), ratios de coûts ($I(R)$). **Rien de tout cela n'est dans Mathlib sous cette forme** : `Mathlib.Combinatorics` ne connaît pas les ensembles d'extensions non équivalentes ; l'entropie et la KL existent (`Mathlib.InformationTheory.KullbackLeibler`) mais pas la KL *empirique à lissage de Laplace* ; le do-calculus n'existe pas. Formaliser `C_t = |AdmExt(G_t)|` en théorie des types exigerait d'abord une définition d'`AdmExt` comme quotient d'extensions à équivalence près — un chantier en soi, pour l'instant hors de portée (et hors de ce dépôt : aucun lake Lean côté IIT).

## Exercices

Les trois exercices suivants manipulent le module `ict.free_coordinates`.

### Exercice 1 — Construire l'équivalence

Construisez une troisième extension de `G0` qui ouvre **exactement** les mêmes actions que `ETA_PM` mais sous un concept différent, puis vérifiez que `non_canonicity` ne bouge pas. Ensuite faites passer $C_t$ à 3 avec une extension réellement distincte.

*Indice : la clé est l'ensemble d'actions résultant, pas le nom du concept.*

In [7]:
def exercice_1():
    # TODO etudiant
    # 1. definir ETA_EQ (concept different, memes actions ouvertes que ETA_PM)
    # 2. verifier : non_canonicity(G0, (ETA_PM, ETA_EQ)) == 1
    # 3. definir ETA_TROIS (actions differentes) et faire monter C_t a 3
    return None

print("Exercice 1 a completer")

Exercice 1 a completer


### Exercice 2 — La température de l'adoption

Faites varier la `temperature` de `simulate_trajectory` (par exemple 0.1, 0.5, 2.0) et mesurez l'institutionnalisation de la personne morale à chaque température. Que se passe-t-il quand les agents deviennent quasi-déterministes, et pourquoi la lecture « innovation vs bruit » dépend-elle de ce paramètre ?

*Indice : à température basse, seuls les payoffs dominants survivent ; à température haute, tout se mélange.*

In [8]:
def exercice_2():
    # TODO etudiant
    # 1. pour temperature dans [0.1, 0.5, 2.0] :
    # 2.   simuler simulate_trajectory(G1_reduit_aux_survivants, horizon=60, rng, temperature=...)
    # 3.   calculer la fraction de coups sur les actions ouvertes par ETA_PM
    # 4. commenter la dependence
    return None

print("Exercice 2 a completer")

Exercice 2 a completer


### Exercice 3 — Le coup capturé

Modifiez les payoffs pour que l'action `signer` ne profite **qu'à alice** (payoff 2.0) et coûte à bob (payoff -1.0). Mesurez $P(R)$ et l'institutionnalisation de ce coup *capturé*. Le contraste avec la personne morale symétrique est-il plus visible sur $P(R)$ ou sur l'institutionnalisation ?

*Indice : un coup capturé peut être performatif (fort $P(R)$) sans être institutionnalisé — c'est exactement la distinction des proxys.*

In [9]:
def exercice_3():
    # TODO etudiant
    # 1. partir de G1 = apply_ontology(G0, ETA_PM) et reecrire les payoffs de 'signer'
    # 2. mesurer performative_power et institutionalization sur ce jeu capture
    # 3. comparer au cas symetrique et commenter
    return None

print("Exercice 3 a completer")

Exercice 3 a completer


## Conclusion

La strate 7 ne livrait que de la prose ; elle a maintenant son **organe exécutable**. Le contraste fondateur est mesuré de bout en bout : l'incertitude sur une *valeur* dans un espace fixé se réduit à des bits (prédicteur bayésien, perte bornée par l'entropie) ; l'incertitude sur *l'espace lui-même* se manifeste comme surprise structurelle — $\log_2(|A|+1)$, la porte que l'espace n'avait pas — et comme un choix réel parmi des extensions non équivalentes ($C_t > 1$), mesuré dans ses effets ($P(R)$), sa persistance (institutionnalisation) et son asymétrie temporelle ($I(R)$).

Ce que ce notebook **ne démontre pas** : ni la physique des freebits d'Aaronson (ordre 1, question empirique ouverte), ni la thèse que la « qualité du choix de coordonnée » est une composante de la performativité (posée sans démonstration au cadrage). Il rend les deux *dicibles* — et c'est le grade honest d'une première livraison de strate.

**Navigation** : précédent dans la série — [ICT-45](ICT-42b-InoculationBifurcation-9B-Python.ipynb) ; le cadrage — [`strate7-cadres-libres.md`](https://github.com/jsboige/CoursIA/blob/main/docs/ict/strate7-cadres-libres.md) ; l'Epic — [#16781](https://github.com/jsboige/CoursIA/issues/16781).